In [ ]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import files
files.upload()  # select kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mahmoudnoor/high-resolution-catdogbird-image-dataset-13000
!unzip -q high-resolution-catdogbird-image-dataset-13000.zip -d catdogbird_data
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d other_data

In [ ]:
!find catdogbird_data -maxdepth 3 -type d
print("---")
!find other_data -maxdepth 3 -type d

In [ ]:
import os

def count_images(path):
    return len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print("cat:", count_images("catdogbird_data/cat/cat"))
print("dog:", count_images("catdogbird_data/dog/dog"))
print("bird:", count_images("catdogbird_data/bird/bird"))

other_classes = ["glacier", "street", "forest", "sea", "mountain", "buildings"]
total_other = 0
for cls in other_classes:
    n_train = count_images(f"other_data/seg_train/seg_train/{cls}")
    total_other += n_train
print("other (seg_train only):", total_other)

In [ ]:
import os, shutil, random

random.seed(42)  # reproducibility

base_dir = "combined_data"
for cls in ["cat", "dog", "bird", "other"]:
    os.makedirs(os.path.join(base_dir, cls), exist_ok=True)

def copy_images(src_dir, dst_dir, limit=None):
    files = [f for f in os.listdir(src_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if limit:
        files = random.sample(files, min(limit, len(files)))
    for f in files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))
    return len(files)

# --- Copy cat, dog, bird as-is (no undersampling needed) ---
n_cat = copy_images("catdogbird_data/cat/cat", f"{base_dir}/cat")
n_dog = copy_images("catdogbird_data/dog/dog", f"{base_dir}/dog")
n_bird = copy_images("catdogbird_data/bird/bird", f"{base_dir}/bird")

# --- Copy "other", undersampled to ~4500 total, spread proportionally across 6 categories ---
other_classes = ["glacier", "street", "forest", "sea", "mountain", "buildings"]
target_other_total = 4500
per_class_limit = target_other_total // len(other_classes)  # 750 each

n_other = 0
for cls in other_classes:
    src = f"other_data/seg_train/seg_train/{cls}"
    # prefix filenames with class name to avoid collisions (e.g., "glacier_1234.jpg")
    files = random.sample(os.listdir(src), min(per_class_limit, len(os.listdir(src))))
    for f in files:
        new_name = f"{cls}_{f}"
        shutil.copy(os.path.join(src, f), os.path.join(f"{base_dir}/other", new_name))
    n_other += len(files)

print(f"cat: {n_cat}, dog: {n_dog}, bird: {n_bird}, other: {n_other}")

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# --- Transforms (same augmentation strategy as before, proven to help) ---
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Load full dataset twice: once with augmentation, once without ---
full_dataset_aug = datasets.ImageFolder("combined_data", transform=train_transform)
full_dataset_eval = datasets.ImageFolder("combined_data", transform=eval_transform)

print("Classes:", full_dataset_aug.classes)  # should print ['bird', 'cat', 'dog', 'other'] (alphabetical)

# --- Split: 70% train, 15% val, 15% test ---
n = len(full_dataset_aug)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

generator = torch.Generator().manual_seed(42)  # reproducible split
train_idx, val_idx, test_idx = random_split(range(n), [n_train, n_val, n_test], generator=generator)

train_dataset = torch.utils.data.Subset(full_dataset_aug, train_idx.indices)
val_dataset = torch.utils.data.Subset(full_dataset_eval, val_idx.indices)
test_dataset = torch.utils.data.Subset(full_dataset_eval, test_idx.indices)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# --- DataLoaders ---
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)   # should be [32, 3, 128, 128]
print("Labels:", labels[:10])          # should show a mix of 0,1,2,3

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MultiAnimalCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)  # <-- changed from 2 to num_classes (4)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiAnimalCNN(num_classes=4).to(device)
print(model)
print("Using device:", device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch():
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def validate():
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

# --- Training loop ---
num_epochs = 15
best_val_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_multi_model.pth")
        marker = " <- saved (best so far)"
    else:
        marker = ""

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Reload best checkpoint
model = MultiAnimalCNN(num_classes=4).to(device)
model.load_state_dict(torch.load("best_multi_model.pth"))
model.eval()

class_names = full_dataset_aug.classes  # e.g. ['bird', 'cat', 'dog', 'other']
print("Class order:", class_names)

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = (all_preds == all_labels).mean()
print(f"\nTest Accuracy: {test_acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Confusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)